**Run evidence** · task `constraints_comments` · run `441384456181939` · result **SUCCESS** · executed 2026-10-19 17:28 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02f_constraints_comments`.

# 02f — PK / FK constraints + column & table COMMENTs (all 5 gold tables)

Unity Catalog supports **informational** primary-key / foreign-key constraints
(declared `NOT ENFORCED`, `RELY`). They power:
* **Genie**: relationship hints so the model joins tables correctly.
* **BI tools & the catalog UI**: ER-diagram rendering & documentation.

We also set rich **table** and **column** comments on every gold table — these are
surfaced to Genie and to any analyst browsing the catalog, and are required by the
spec ("column descriptions and table descriptions").

Order of operations for each PK:
1. `ALTER TABLE ... ALTER COLUMN ... SET NOT NULL` on every key column.
2. `ALTER TABLE ... ADD CONSTRAINT ... PRIMARY KEY (...)`.
3. Add FKs referencing the parent PKs.

**SUPPLIER_RISK** is a materialized view owned by the Lakeflow pipeline (Workflow D,
`02d_supplier_risk_pipeline.sql`). Its NOT NULL columns, primary key
`pk_risk (LIFNR, VALID_FROM)` and table / column comments are declared in the
pipeline definition, because a pipeline-owned table cannot be altered here. The
verification cell at the end lists the constraints of all 5 gold tables.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
NS = CFG.namespace


def run(sql):
    print("→", sql)
    spark.sql(sql)


## Idempotency helper — drop existing constraints before re-adding
Lets this notebook be re-run safely.

In [ ]:

def drop_constraints(table, names):
    for n in names:
        try:
            spark.sql(f"ALTER TABLE {NS}.{table} DROP CONSTRAINT IF EXISTS {n}")
        except Exception as e:
            print(f"  (skip drop {n}: {e})")


# Child tables first: a PK cannot be dropped while a FK still references it.
for t, cons in {
    "SOURCING": ["fk_sourcing_material", "fk_sourcing_supplier", "pk_sourcing"],
    "PURCHASE_ORDER": ["fk_po_supplier", "pk_po"],
    "MATERIAL": ["pk_material"],
    "SUPPLIER": ["pk_supplier"],
}.items():
    drop_constraints(t, cons)

## 1) NOT NULL on all key columns (required before a PK can be declared)

In [ ]:
for tbl, cols in {
    "MATERIAL": ["MATNR", "WERKS"],
    "SUPPLIER": ["LIFNR", "EKORG"],
    "SOURCING": ["MATNR", "WERKS", "LIFNR"],
    "PURCHASE_ORDER": ["EBELN", "EBELP"],
}.items():
    for c in cols:
        run(f"ALTER TABLE {NS}.{tbl} ALTER COLUMN {c} SET NOT NULL")

→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN MATNR SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN WERKS SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SUPPLIER ALTER COLUMN LIFNR SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SUPPLIER ALTER COLUMN EKORG SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING ALTER COLUMN MATNR SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING ALTER COLUMN WERKS SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING ALTER COLUMN LIFNR SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.PURCHASE_ORDER ALTER COLUMN EBELN SET NOT NULL
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.PURCHASE_ORDER ALTER COLUMN EBELP SET NOT NULL


## 2) Primary keys (informational, NOT ENFORCED, RELY)

In [ ]:
run(f"ALTER TABLE {NS}.MATERIAL ADD CONSTRAINT pk_material PRIMARY KEY (MATNR, WERKS)")
run(f"ALTER TABLE {NS}.SUPPLIER ADD CONSTRAINT pk_supplier PRIMARY KEY (LIFNR, EKORG)")
run(f"ALTER TABLE {NS}.SOURCING ADD CONSTRAINT pk_sourcing PRIMARY KEY (MATNR, WERKS, LIFNR)")
# PURCHASE_ORDER PK on EBELN+EBELP.
run(f"ALTER TABLE {NS}.PURCHASE_ORDER ADD CONSTRAINT pk_po PRIMARY KEY (EBELN, EBELP)")
# SUPPLIER_RISK (time-sliced, PK LIFNR+VALID_FROM) gets its PK in the Lakeflow pipeline.

→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ADD CONSTRAINT pk_material PRIMARY KEY (MATNR, WERKS)
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SUPPLIER ADD CONSTRAINT pk_supplier PRIMARY KEY (LIFNR, EKORG)
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING ADD CONSTRAINT pk_sourcing PRIMARY KEY (MATNR, WERKS, LIFNR)
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.PURCHASE_ORDER ADD CONSTRAINT pk_po PRIMARY KEY (EBELN, EBELP)


## 3) Foreign keys
* SOURCING.(MATNR,WERKS) → MATERIAL   (each sourcing row is for a real material+plant)
* SOURCING.LIFNR references a supplier (we FK to SUPPLIER on the LIFNR portion via a
  composite is not possible since SUPPLIER PK is LIFNR+EKORG; instead we document
  the relationship in comments and add a FK where the grain matches).
* PURCHASE_ORDER.(MATNR is a material ref); LIFNR is a supplier ref.

UC FKs must reference the full PK of the parent. Where the child does not carry the
full parent key (e.g. SOURCING has LIFNR but not EKORG), we cannot declare a strict
FK; we capture that relationship in table comments for Genie instead. The FKs below
are the ones whose child columns fully cover a parent PK.

In [ ]:
# SOURCING (MATNR,WERKS) fully covers MATERIAL's PK → valid FK.
run(f"""ALTER TABLE {NS}.SOURCING ADD CONSTRAINT fk_sourcing_material
        FOREIGN KEY (MATNR, WERKS) REFERENCES {NS}.MATERIAL (MATNR, WERKS)""")

print("FK SOURCING→MATERIAL added. "
      "SOURCING.LIFNR↔SUPPLIER.LIFNR and PURCHASE_ORDER.LIFNR↔SUPPLIER.LIFNR "
      "relationships are documented in comments (parent PK includes EKORG, so no strict FK).")

→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING ADD CONSTRAINT fk_sourcing_material
        FOREIGN KEY (MATNR, WERKS) REFERENCES max_kohler_demo_catalog.purchasing_demo.MATERIAL (MATNR, WERKS)
FK SOURCING→MATERIAL added. SOURCING.LIFNR↔SUPPLIER.LIFNR and PURCHASE_ORDER.LIFNR↔SUPPLIER.LIFNR relationships are documented in comments (parent PK includes EKORG, so no strict FK).


## 4) Table comments

In [ ]:
TABLE_COMMENTS = {
    "MATERIAL":
        "Gold: material master consolidated to one row per material (MATNR) and plant "
        "(WERKS). Built by Workflow A from RAW_MARC + RAW_MARA + RAW_MAKT (EN only). "
        "Answers 'what is this material, what group/type, which plant'.",
    "SUPPLIER":
        "Gold: vendor master consolidated to one row per vendor (LIFNR) and purchasing "
        "org (EKORG). Built by Workflow B from RAW_LFM1 + RAW_LFA1. SPERM='X' means the "
        "vendor holds a purchasing block in that org.",
    "SOURCING":
        "Gold: approved sourcing relationships at grain material+plant+vendor "
        "(MATNR+WERKS+LIFNR). Built by Workflow C from RAW_EORD + flattened quota "
        "(RAW_EQUK⨝RAW_EQUP). QUOTA may be NULL (source-listed but not quota-split — a "
        "meaningful gap). IS_VALID_SOURCE=true means a non-blocked source-list entry "
        "(NOTKZ<>'X'). Central table for multi-sourcing / single-sourcing analysis.",
    "PURCHASE_ORDER":
        "Gold: purchase-order lines at grain PO number + item (EBELN+EBELP). Built by "
        "Workflow E from RAW_EKPO⨝RAW_EKKO with earliest goods-receipt date from "
        "RAW_EKBE. ON_TIME_FLAG compares GR_DATE to REQUESTED_DATE. Used for on-time "
        "delivery, net-price variance, and orders-under-active-risk analysis.",
}
for t, c in TABLE_COMMENTS.items():
    run(f"COMMENT ON TABLE {NS}.{t} IS '{c.replace(chr(39), chr(39)+chr(39))}'")

→ COMMENT ON TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL IS 'Gold: material master consolidated to one row per material (MATNR) and plant (WERKS). Built by Workflow A from RAW_MARC + RAW_MARA + RAW_MAKT (EN only). Answers ''what is this material, what group/type, which plant''.'
→ COMMENT ON TABLE max_kohler_demo_catalog.purchasing_demo.SUPPLIER IS 'Gold: vendor master consolidated to one row per vendor (LIFNR) and purchasing org (EKORG). Built by Workflow B from RAW_LFM1 + RAW_LFA1. SPERM=''X'' means the vendor holds a purchasing block in that org.'
→ COMMENT ON TABLE max_kohler_demo_catalog.purchasing_demo.SOURCING IS 'Gold: approved sourcing relationships at grain material+plant+vendor (MATNR+WERKS+LIFNR). Built by Workflow C from RAW_EORD + flattened quota (RAW_EQUK⨝RAW_EQUP). QUOTA may be NULL (source-listed but not quota-split — a meaningful gap). IS_VALID_SOURCE=true means a non-blocked source-list entry (NOTKZ<>''X''). Central table for multi-sourcing / single-sourci

## 5) Column comments (every column on every gold table)

In [ ]:
COLUMN_COMMENTS = {
    "MATERIAL": {
        "MATNR": "Material number (SAP MARA/MARC key).",
        "WERKS": "Plant code where the material is procured/produced.",
        "MAKTX": "Material short description (English, SPRAS='EN').",
        "MTART": "Material type: ROH=raw, HALB=semi-finished, FERT=finished.",
        "MATKL": "Material group (e.g. ELEC, MECH, CHEM, ELAST, FAST).",
        "DISPO": "MRP controller responsible for the material at this plant.",
        "SOBSL": "Special procurement key (blank=standard).",
    },
    "SUPPLIER": {
        "LIFNR": "Vendor (supplier) number.",
        "EKORG": "Purchasing organization the vendor is set up in.",
        "NAME1": "Vendor legal name.",
        "LAND1": "Vendor country ISO code (e.g. DE, FR, CN).",
        "SPERM": "Purchasing block flag: 'X' = vendor blocked for purchasing in this org.",
    },
    "SOURCING": {
        "MATNR": "Material number (FK to MATERIAL).",
        "WERKS": "Plant code (FK to MATERIAL).",
        "LIFNR": "Approved vendor (relates to SUPPLIER.LIFNR).",
        "VDATU": "Source-list validity start date.",
        "BDATU": "Source-list validity end date.",
        "NOTKZ": "Blocked-source indicator: 'X' = this source-list entry is blocked.",
        "FLIFN": "Fixed-vendor flag: 'X' = preferred/fixed source.",
        "QUOTA": "Quota percentage allocated to this vendor (NULL if not quota-split).",
        "IS_VALID_SOURCE": "TRUE when NOTKZ<>'X' (a valid, non-blocked source).",
    },
    "PURCHASE_ORDER": {
        "EBELN": "Purchase order number.",
        "EBELP": "Purchase order line item number.",
        "MATNR": "Ordered material (relates to MATERIAL.MATNR).",
        "LIFNR": "Vendor the PO was placed with (relates to SUPPLIER.LIFNR).",
        "EKORG": "Purchasing organization of the PO.",
        "NETPR": "Net price per unit on the PO line.",
        "ORDER_DATE": "PO creation/order date (SAP EKKO.AEDAT).",
        "REQUESTED_DATE": "Requested delivery date (SAP EKPO.EINDT).",
        "GR_DATE": "Earliest goods-receipt date (NULL = not yet received).",
        "ON_TIME_FLAG": "TRUE if GR_DATE<=REQUESTED_DATE, FALSE if late, NULL if no receipt.",
    },
}
for t, cols in COLUMN_COMMENTS.items():
    for col, cmt in cols.items():
        run(f"ALTER TABLE {NS}.{t} ALTER COLUMN {col} "
            f"COMMENT '{cmt.replace(chr(39), chr(39)+chr(39))}'")

→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN MATNR COMMENT 'Material number (SAP MARA/MARC key).'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN WERKS COMMENT 'Plant code where the material is procured/produced.'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN MAKTX COMMENT 'Material short description (English, SPRAS=''EN'').'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN MTART COMMENT 'Material type: ROH=raw, HALB=semi-finished, FERT=finished.'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN MATKL COMMENT 'Material group (e.g. ELEC, MECH, CHEM, ELAST, FAST).'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN DISPO COMMENT 'MRP controller responsible for the material at this plant.'
→ ALTER TABLE max_kohler_demo_catalog.purchasing_demo.MATERIAL ALTER COLUMN SOBSL COMMENT 'Special procurement key (blank=standard).'
→ 

## Verify constraints & comments

In [ ]:
cons = spark.sql(f"""
  SELECT tc.table_name, tc.constraint_name, tc.constraint_type,
         concat_ws(', ', collect_list(kcu.column_name)) AS columns
  FROM {CFG.catalog}.information_schema.table_constraints tc
  JOIN {CFG.catalog}.information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name AND tc.table_schema = kcu.table_schema
  WHERE tc.table_schema = '{CFG.schema}'
  GROUP BY ALL ORDER BY tc.table_name, tc.constraint_type DESC""")
display(cons)
pks = {r["table_name"].upper() for r in cons.collect() if r["constraint_type"] == "PRIMARY KEY"}
missing = [t for t in GOLD_TABLES if t not in pks]
print(f"Gold tables with a primary key: {sorted(pks)}")
assert not missing, f"Missing primary key on: {missing}"

for t in GOLD_TABLES:
    print(f"\n===== {t} =====")
    spark.sql(f"DESCRIBE TABLE EXTENDED {NS}.{t}").show(60, truncate=False)

table_name,constraint_name,constraint_type,columns
material,pk_material,PRIMARY KEY,"WERKS, MATNR"
purchase_order,pk_po,PRIMARY KEY,"EBELN, EBELP"
risk_worklist,risk_worklist_pk,PRIMARY KEY,"WERKS, MATNR"
sourcing,pk_sourcing,PRIMARY KEY,"LIFNR, MATNR, WERKS"
sourcing,fk_sourcing_material,FOREIGN KEY,"MATNR, WERKS"
supplier,pk_supplier,PRIMARY KEY,"LIFNR, EKORG"
supplier_risk,pk_risk,PRIMARY KEY,"VALID_FROM, LIFNR"


Gold tables with a primary key: ['MATERIAL', 'PURCHASE_ORDER', 'RISK_WORKLIST', 'SOURCING', 'SUPPLIER', 'SUPPLIER_RISK']

===== MATERIAL =====
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------+
|col_name                    |data_type                                                                                                                                                                                                                                                                                                                   |comment                                                   |
+----------------------------+-------------

In [ ]:
dbutils.notebook.exit("02f_constraints_comments: SUCCESS — PK/FK + comments applied.")

Notebook exited: 02f_constraints_comments: SUCCESS — PK/FK + comments applied.
